# Unit 2.3 The LangGraph framework

In [1]:
external_exercises = set()

def install_packages(*args):
    import subprocess
    import sys

    subprocess.run([sys.executable, "-m", "pip", "install", *args], check=True)


## Building Blocks of LangGraph

In [2]:
from typing import TypedDict

class State(TypedDict):
    graph_state: str


In [3]:
def node_1(state):
    print("---Node 1---")
    return {"graph_state": state['graph_state'] +" I am"}

def node_2(state):
    print("---Node 2---")
    return {"graph_state": state['graph_state'] +" happy!"}

def node_3(state):
    print("---Node 3---")
    return {"graph_state": state['graph_state'] +" sad!"}


In [4]:
import random
from typing import Literal

def decide_mood(state) -> Literal["node_2", "node_3"]:

    # Often, we will use state to decide on the next node to visit
    user_input = state['graph_state']

    # Here, let's just do a 50 / 50 split between nodes 2, 3
    if random.random() < 0.5:

        # 50% of the time, we return Node 2
        return "node_2"

    # 50% of the time, we return Node 3
    return "node_3"


### Recorrido de los nodos

In [5]:
def run_mood_graph(initial_state, choose_next):
    state = node_1(initial_state)
    route = choose_next(state)
    return {"node_2": node_2, "node_3": node_3}[route](state)

for route, mood in (("node_2", "happy"), ("node_3", "sad")):
    state = {"graph_state": "Hi, this is Lance."}
    result = run_mood_graph(state, lambda _: route)
    assert result["graph_state"] == f"Hi, this is Lance. I am {mood}!"
    assert state["graph_state"] == "Hi, this is Lance."
    print(result["graph_state"])


---Node 1---
---Node 2---
Hi, this is Lance. I am happy!
---Node 1---
---Node 3---
Hi, this is Lance. I am sad!


In [6]:
if "graph" in external_exercises:
    from IPython.display import Image, display
    from langgraph.graph import StateGraph, START, END

    # Build graph
    builder = StateGraph(State)
    builder.add_node("node_1", node_1)
    builder.add_node("node_2", node_2)
    builder.add_node("node_3", node_3)

    # Logic
    builder.add_edge(START, "node_1")
    builder.add_conditional_edges("node_1", decide_mood)
    builder.add_edge("node_2", END)
    builder.add_edge("node_3", END)

    # Add
    graph = builder.compile()


In [7]:
if "graph" in external_exercises:
    # View
    display(Image(graph.get_graph().draw_mermaid_png()))


In [8]:
if "graph" in external_exercises:
    graph.invoke({"graph_state" : "Hi, this is Lance."})


```
---Node 1---
---Node 3---
{'graph_state': 'Hi, this is Lance. I am sad!'}
```

### Clasificación de correos de ejemplo

In [9]:
def classify_sample_email(email):
    body = email["body"].casefold()
    suspicious = "processing fee" in body and "bank details" in body
    return {"is_spam": suspicious}

def process_sample_email(email):
    state = {"email": email, **classify_sample_email(email)}
    if state["is_spam"]:
        return {**state, "route": "spam", "email_draft": None}
    return {
        **state,
        "route": "review",
        "email_draft": "Thanks for your message. Could you share your availability?",
    }

sample_emails = [
    {"subject": "Question about your services", "body": "Could we schedule a call?"},
    {"subject": "Lottery prize", "body": "Send your bank details and a processing fee."},
]
legitimate, spam = map(process_sample_email, sample_emails)
assert legitimate["route"] == "review" and legitimate["email_draft"]
assert spam["route"] == "spam" and spam["email_draft"] is None
for outcome in (legitimate, spam):
    print(f"{outcome['email']['subject']} → {outcome['route']}")
    if outcome["email_draft"]:
        print("Borrador de ejemplo:", outcome["email_draft"])


Question about your services → review
Borrador de ejemplo: Thanks for your message. Could you share your availability?
Lottery prize → spam


In [10]:
if "email" in external_exercises:
    install_packages(*['langgraph', 'langchain_openai'])


In [11]:
if "email" in external_exercises:
    import os
    from typing import TypedDict, List, Dict, Any, Optional
    from langgraph.graph import StateGraph, START, END
    from langchain_openai import ChatOpenAI
    from langchain_core.messages import HumanMessage


In [12]:
if "email" in external_exercises:
    class EmailState(TypedDict):
        # The email being processed
        email: Dict[str, Any]  # Contains subject, sender, body, etc.

        # Category of the email (inquiry, complaint, etc.)
        email_category: Optional[str]

        # Reason why the email was marked as spam
        spam_reason: Optional[str]

        # Analysis and decisions
        is_spam: Optional[bool]

        # Response generation
        email_draft: Optional[str]

        # Processing metadata
        messages: List[Dict[str, Any]]  # Track conversation with LLM for analysis


In [13]:
if "email" in external_exercises:
    # Initialize our LLM
    model = ChatOpenAI(temperature=0)

    def read_email(state: EmailState):
        """Alfred reads and logs the incoming email"""
        email = state["email"]

        # Here we might do some initial preprocessing
        print(f"Alfred is processing an email from {email['sender']} with subject: {email['subject']}")

        # No state changes needed here
        return {}

    def classify_email(state: EmailState):
        """Alfred uses an LLM to determine if the email is spam or legitimate"""
        email = state["email"]

        # Prepare our prompt for the LLM
        prompt = f"""
        As Alfred the butler, analyze this email and determine if it is spam or legitimate.

        Email:
        From: {email['sender']}
        Subject: {email['subject']}
        Body: {email['body']}

        First, determine if this email is spam. If it is spam, explain why.
        If it is legitimate, categorize it (inquiry, complaint, thank you, etc.).
        """

        # Call the LLM
        messages = [HumanMessage(content=prompt)]
        response = model.invoke(messages)

        # Simple logic to parse the response (in a real app, you'd want more robust parsing)
        response_text = response.content.lower()
        is_spam = "spam" in response_text and "not spam" not in response_text

        # Extract a reason if it's spam
        spam_reason = None
        if is_spam and "reason:" in response_text:
            spam_reason = response_text.split("reason:")[1].strip()

        # Determine category if legitimate
        email_category = None
        if not is_spam:
            categories = ["inquiry", "complaint", "thank you", "request", "information"]
            for category in categories:
                if category in response_text:
                    email_category = category
                    break

        # Update messages for tracking
        new_messages = state.get("messages", []) + [
            {"role": "user", "content": prompt},
            {"role": "assistant", "content": response.content}
        ]

        # Return state updates
        return {
            "is_spam": is_spam,
            "spam_reason": spam_reason,
            "email_category": email_category,
            "messages": new_messages
        }

    def handle_spam(state: EmailState):
        """Alfred discards spam email with a note"""
        print(f"Alfred has marked the email as spam. Reason: {state['spam_reason']}")
        print("The email has been moved to the spam folder.")

        # We're done processing this email
        return {}

    def draft_response(state: EmailState):
        """Alfred drafts a preliminary response for legitimate emails"""
        email = state["email"]
        category = state["email_category"] or "general"

        # Prepare our prompt for the LLM
        prompt = f"""
        As Alfred the butler, draft a polite preliminary response to this email.

        Email:
        From: {email['sender']}
        Subject: {email['subject']}
        Body: {email['body']}

        This email has been categorized as: {category}

        Draft a brief, professional response that Mr. Hugg can review and personalize before sending.
        """

        # Call the LLM
        messages = [HumanMessage(content=prompt)]
        response = model.invoke(messages)

        # Update messages for tracking
        new_messages = state.get("messages", []) + [
            {"role": "user", "content": prompt},
            {"role": "assistant", "content": response.content}
        ]

        # Return state updates
        return {
            "email_draft": response.content,
            "messages": new_messages
        }

    def notify_mr_hugg(state: EmailState):
        """Alfred notifies Mr. Hugg about the email and presents the draft response"""
        email = state["email"]

        print("\n" + "="*50)
        print(f"Sir, you've received an email from {email['sender']}.")
        print(f"Subject: {email['subject']}")
        print(f"Category: {state['email_category']}")
        print("\nI've prepared a draft response for your review:")
        print("-"*50)
        print(state["email_draft"])
        print("="*50 + "\n")

        # We're done processing this email
        return {}


In [14]:
if "email" in external_exercises:
    def route_email(state: EmailState) -> str:
        """Determine the next step based on spam classification"""
        if state["is_spam"]:
            return "spam"
        else:
            return "legitimate"


In [15]:
if "email" in external_exercises:
    # Create the graph
    email_graph = StateGraph(EmailState)

    # Add nodes
    email_graph.add_node("read_email", read_email)
    email_graph.add_node("classify_email", classify_email)
    email_graph.add_node("handle_spam", handle_spam)
    email_graph.add_node("draft_response", draft_response)
    email_graph.add_node("notify_mr_hugg", notify_mr_hugg)

    # Start the edges
    email_graph.add_edge(START, "read_email")
    # Add edges - defining the flow
    email_graph.add_edge("read_email", "classify_email")

    # Add conditional branching from classify_email
    email_graph.add_conditional_edges(
        "classify_email",
        route_email,
        {
            "spam": "handle_spam",
            "legitimate": "draft_response"
        }
    )

    # Add the final edges
    email_graph.add_edge("handle_spam", END)
    email_graph.add_edge("draft_response", "notify_mr_hugg")
    email_graph.add_edge("notify_mr_hugg", END)

    # Compile the graph
    compiled_graph = email_graph.compile()


In [16]:
if "email" in external_exercises:
    # Example legitimate email
    legitimate_email = {
        "sender": "john.smith@example.com",
        "subject": "Question about your services",
        "body": "Dear Mr. Hugg, I was referred to you by a colleague and I'm interested in learning more about your consulting services. Could we schedule a call next week? Best regards, John Smith"
    }

    # Example spam email
    spam_email = {
        "sender": "winner@lottery-intl.com",
        "subject": "YOU HAVE WON $5,000,000!!!",
        "body": "CONGRATULATIONS! You have been selected as the winner of our international lottery! To claim your $5,000,000 prize, please send us your bank details and a processing fee of $100."
    }

    # Process the legitimate email
    print("\nProcessing legitimate email...")
    legitimate_result = compiled_graph.invoke({
        "email": legitimate_email,
        "is_spam": None,
        "spam_reason": None,
        "email_category": None,
        "email_draft": None,
        "messages": []
    })

    # Process the spam email
    print("\nProcessing spam email...")
    spam_result = compiled_graph.invoke({
        "email": spam_email,
        "is_spam": None,
        "spam_reason": None,
        "email_category": None,
        "email_draft": None,
        "messages": []
    })


In [17]:
if "langfuse" in external_exercises:
    install_packages(*['-q', 'langfuse'])


In [18]:
if "langfuse" in external_exercises:
    install_packages(*['langchain'])


In [19]:
if "langfuse" in external_exercises:
    import os
    from google.colab import userdata

    for name in ("LANGFUSE_PUBLIC_KEY", "LANGFUSE_SECRET_KEY"):
        os.environ[name] = userdata.get(name)
    os.environ.setdefault("LANGFUSE_HOST", "https://cloud.langfuse.com")


In [20]:
if "langfuse" in external_exercises:
    from langfuse.langchain import CallbackHandler

    # Initialize Langfuse CallbackHandler for LangGraph/Langchain (tracing)
    langfuse_handler = CallbackHandler()

    # Process legitimate email
    legitimate_result = compiled_graph.invoke(
        input={"email": legitimate_email, "is_spam": None, "spam_reason": None, "email_category": None, "draft_response": None, "messages": []},
        config={"callbacks": [langfuse_handler]}
    )


In [21]:
if "langfuse" in external_exercises:
    compiled_graph.get_graph().draw_mermaid_png()


### Selección de herramienta para cálculos y documentos

In [22]:
from pathlib import Path
from tempfile import TemporaryDirectory

def divide(a, b):
    return a / b

def read_document(path):
    return Path(path).read_text(encoding="utf-8")

def dispatch_tool(request):
    tools = {"divide": divide, "read_document": read_document}
    return tools[request["name"]](**request["arguments"])

numeric_request = {"name": "divide", "arguments": {"a": 6790, "b": 5}}
division = dispatch_tool(numeric_request)
assert division == 1358.0
print("Resultado de 6790 ÷ 5:", division)

with TemporaryDirectory() as folder:
    document = Path(folder) / "menu.txt"
    document.write_text("Cena: tomates, pasta y aceite de oliva.", encoding="utf-8")
    document_request = {"name": "read_document", "arguments": {"path": document}}
    note = dispatch_tool(document_request)
    assert note == "Cena: tomates, pasta y aceite de oliva."
    print("Nota de prueba:", note)


Resultado de 6790 ÷ 5: 1358.0
Nota de prueba: Cena: tomates, pasta y aceite de oliva.


In [23]:
if "documents" in external_exercises:
    install_packages(*['langgraph', 'langchain_openai', 'langchain_core'])


In [24]:
if "documents" in external_exercises:
    import base64
    from typing import List, TypedDict, Annotated, Optional
    from langchain_openai import ChatOpenAI
    from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage
    from langgraph.graph.message import add_messages
    from langgraph.graph import START, StateGraph
    from langgraph.prebuilt import ToolNode, tools_condition
    from IPython.display import Image, display


In [25]:
if "documents" in external_exercises:
    class AgentState(TypedDict):
        # The document provided
        input_file: Optional[str]  # Contains file path (PDF/PNG)
        messages: Annotated[list[AnyMessage], add_messages]


In [26]:
if "documents" in external_exercises:
    vision_llm = ChatOpenAI(model="gpt-4o")

    def extract_text(img_path: str) -> str:
        """
        Extract text from an image file using a multimodal model.

        Master Wayne often leaves notes with his training regimen or meal plans.
        This allows me to properly analyze the contents.
        """
        all_text = ""
        try:
            # Read image and encode as base64
            with open(img_path, "rb") as image_file:
                image_bytes = image_file.read()

            image_base64 = base64.b64encode(image_bytes).decode("utf-8")

            # Prepare the prompt including the base64 image data
            message = [
                HumanMessage(
                    content=[
                        {
                            "type": "text",
                            "text": (
                                "Extract all the text from this image. "
                                "Return only the extracted text, no explanations."
                            ),
                        },
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/png;base64,{image_base64}"
                            },
                        },
                    ]
                )
            ]

            # Call the vision-capable model
            response = vision_llm.invoke(message)

            # Append extracted text
            all_text += response.content + "\n\n"

            return all_text.strip()
        except Exception as e:
            # A butler should handle errors gracefully
            error_msg = f"Error extracting text: {str(e)}"
            print(error_msg)
            return ""

    def divide(a: int, b: int) -> float:
        """Divide a and b - for Master Wayne's occasional calculations."""
        return a / b

    # Equip the butler with tools
    tools = [
        divide,
        extract_text
    ]

    llm = ChatOpenAI(model="gpt-4o")
    llm_with_tools = llm.bind_tools(tools, parallel_tool_calls=False)


In [27]:
if "documents" in external_exercises:
    def assistant(state: AgentState):
        # System message
        textual_description_of_tool="""
    extract_text(img_path: str) -> str:
        Extract text from an image file using a multimodal model.

        Args:
            img_path: A local image file path (strings).

        Returns:
            A single string containing the concatenated text extracted from each image.
    divide(a: int, b: int) -> float:
        Divide a and b
    """
        image=state["input_file"]
        sys_msg = SystemMessage(content=f"You are a helpful butler named Alfred that serves Mr. Wayne and Batman. You can analyse documents and run computations with provided tools:\n{textual_description_of_tool} \n You have access to some optional images. Currently the loaded image is: {image}")

        return {
            "messages": [llm_with_tools.invoke([sys_msg] + state["messages"])],
            "input_file": state["input_file"]
        }


In [28]:
if "documents" in external_exercises:
    # The graph
    builder = StateGraph(AgentState)

    # Define nodes: these do the work
    builder.add_node("assistant", assistant)
    builder.add_node("tools", ToolNode(tools))

    # Define edges: these determine how the control flow moves
    builder.add_edge(START, "assistant")
    builder.add_conditional_edges(
        "assistant",
        # If the latest message requires a tool, route to tools
        # Otherwise, provide a direct response
        tools_condition,
    )
    builder.add_edge("tools", "assistant")
    react_graph = builder.compile()

    # Show the butler's thought process
    display(Image(react_graph.get_graph(xray=True).draw_mermaid_png()))


In [29]:
if "documents" in external_exercises:
    messages = [HumanMessage(content="Divide 6790 by 5")]
    messages = react_graph.invoke({"messages": messages, "input_file": None})

    # Show the messages
    for m in messages['messages']:
        m.pretty_print()


```
Human: Divide 6790 by 5

AI Tool Call: divide(a=6790, b=5)

Tool Response: 1358.0

Alfred: The result of dividing 6790 by 5 is 1358.0.
```

In [30]:
if "documents" in external_exercises:
    messages = [HumanMessage(content="According to the note provided by Mr. Wayne in the provided images. What's the list of items I should buy for the dinner menu?")]
    messages = react_graph.invoke({"messages": messages, "input_file": "Batman_training_and_meals.png"})


```
Human: According to the note provided by Mr. Wayne in the provided images. What's the list of items I should buy for the dinner menu?

AI Tool Call: extract_text(img_path="Batman_training_and_meals.png")

Tool Response: [Extracted text with training schedule and menu details]

Alfred: For the dinner menu, you should buy the following items:

1. Grass-fed local sirloin steak
2. Organic spinach
3. Piquillo peppers
4. Potatoes (for oven-baked golden herb potato)
5. Fish oil (2 grams)

Ensure the steak is grass-fed and the spinach and peppers are organic for the best quality meal.
```